# Canonical Correlation Analysis of 16S V1-V3 and V4 and metabolomics table with top VIPs

Date created: 1/29/2024

Notebook author: Yang Chen

Data analysis by: Britta de Pessemier, Yang Chen

In [1092]:
import biom
from biom import load_table
from biom.util import biom_open
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from skbio.stats.composition import clr
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

In [1093]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    # Transpose df
    df = df.T
    
    return df


In [1094]:
def save_as_biom(df: pd.DataFrame, output_path: str):
    """
    Save a pandas DataFrame as a BIOM table.

    Parameters:
    df (pd.DataFrame): The DataFrame to save.
    output_path (str): Path to the output BIOM file.
    """
    table = biom.table.Table(df.values, observation_ids=df.index, sample_ids=df.columns)
    with biom_open(output_path, 'w') as f:
        table.to_hdf5(f, "save biom")

In [1095]:
# Read in the metadata
metadata = pd.read_csv('../Metadata/metadata_final_18062024.tsv', sep='\t')
metadata

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,inflammatory_lesions_face,noninflammatory_lesions_face,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group
0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,...,0,0,33.765638,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,36,72,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,26,69,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,0,0,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,13,90,28.819451,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,19,77,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,0,0,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,0,0,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,31,55,22.494605,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L


In [1096]:
# skin_group = "Acne_L"
# skin_group = "Acne_NL"
skin_group = "Healthy"

In [1097]:
# Read in metabolomics table and set index
metaB_tbl = pd.read_csv('../Data/metabolomics/Run3_10252024/output/data_sample_10282024.csv')
metaB_tbl = metaB_tbl.set_index('SampleID')

# Read in VIP-filtered feature list
VIP_filtered = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_with_shortened_names.tsv', sep='\t')

# Get list of VIP feature IDs
vip_features = VIP_filtered['ID'].astype(str).unique().tolist()

# Extract the feature ID from each column name (assumes feature ID is before the first semicolon)
col_ids = [col.split(';')[0] for col in metaB_tbl.columns]

# Map original column names to their extracted feature IDs
col_map = dict(zip(metaB_tbl.columns, col_ids))

# Keep only columns where the extracted ID matches a VIP feature
cols_to_keep = [col for col, cid in col_map.items() if cid in vip_features]

# Subset the table to keep only those columns
metaB_tbl = metaB_tbl[cols_to_keep]

# Rename columns using Shortened_Compound_Name from VIP_filtered
# Create mapping of ID to Shortened_Compound_Name
name_map = dict(zip(VIP_filtered['ID'].astype(str), VIP_filtered['Shortened_Compound_Name']))

# Create a mapping from feature ID to shortened name, mz, and RT
VIP_filtered['ID'] = VIP_filtered['ID'].astype(str)  # Ensure IDs are strings
name_map = VIP_filtered.set_index('ID')['Shortened_Compound_Name'].to_dict()
mz_map = VIP_filtered.set_index('ID')['mz'].to_dict()
rt_map = VIP_filtered.set_index('ID')['RT'].to_dict()

# Build new column names
new_cols = []
for col in metaB_tbl.columns:
    feature_id = col.split(';')[0]  # Extract feature ID
    short_name = name_map.get(feature_id, feature_id)
    mz = mz_map.get(feature_id, 'NA')
    rt = rt_map.get(feature_id, 'NA')
    new_col_name = f"{short_name} (mz: {mz}, RT: {rt})"
    new_cols.append(new_col_name)

# Rename the columns
metaB_tbl.columns = new_cols

# View the result
metaB_tbl


,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD001.D0.C1,2987383.80,2025607.40,29574.6480,39859.7580,28277.594,0.00,970487.500,1942604.500,567510.060,1473068.10,...,62007.1100,12406.8260,611709.700,16778.838,0.0000,0.000,0.000,0.00,0.0000,125692.3300
LAMI.RD306.D9.C2,2051791.50,2260333.20,34466.4840,0.0000,253083.810,178911.47,1071868.200,3572847.500,672054.400,308019.97,...,153381.2700,20433.4900,437938.200,12865.390,0.0000,0.000,0.000,0.00,0.0000,10553.9270
LAMI.RD308.D2.C2,1121916.40,1669351.50,2905.5796,3065.1714,19280.236,0.00,750870.100,1618142.800,442653.660,286004.00,...,87817.2000,5348.9400,231525.840,14299.352,0.0000,0.000,0.000,0.00,3822.6377,4734.5693
LAMI.RD304.D11.C1,1818308.60,1434033.20,47379.2730,36750.4000,15391.807,0.00,595568.940,1605551.000,328151.120,299140.06,...,58588.9260,9403.6520,78491.530,0.000,5375.8525,11225.194,23546.951,99470.69,29824.6270,12473.0920
LAMI.RD304.D11.C2,1126503.20,706197.75,3460.0974,0.0000,15015.189,0.00,303845.300,865753.500,152392.720,191713.67,...,20153.3320,2286.2397,261996.800,15780.309,2691.0256,16378.752,34863.656,157503.92,44084.5860,40673.1100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,1173415.00,1250164.60,8613.8590,9787.1210,14712.559,0.00,579058.800,1380958.500,498893.780,236111.75,...,24563.9240,5144.6616,122958.260,56654.043,0.0000,0.000,0.000,0.00,0.0000,0.0000
LAMI.RD308.D9.C3,550094.94,687527.06,3163.9185,3076.9688,17934.553,0.00,281896.700,742416.600,262886.470,303509.40,...,24480.3900,0.0000,65659.700,21826.117,0.0000,0.000,0.000,0.00,0.0000,220846.1600
LAMI.RD319.D2.C2,602147.10,538412.94,4006.6484,0.0000,10556.326,0.00,255542.720,1194061.900,121803.890,5406879.00,...,35224.5800,0.0000,73231.560,19741.246,0.0000,0.000,0.000,0.00,0.0000,25695.0140


In [1098]:
# Read in 16S V1-V3 table
V1V3_biom_path = '../Data/16S/Tables/16S_V1-V3_Genus_collapsed_absolute.biom'
V1V3_tbl = biom_to_df(V1V3_biom_path)
# Extract the feature name after the last semicolon for each column
# V1V3_tbl.columns = V1V3_tbl.columns.str.split(';').str[-1].str.strip()
# # Sort columns by their sum in descending order
# V1V3_tbl = V1V3_tbl.loc[:, V1V3_tbl.sum().sort_values(ascending=False).index]


# Read in 16S V4 table
V4_biom_path = '../Data/16S/Tables/16S_V4_Genus_collapsed_absolute.biom'
V4_tbl = biom_to_df(V4_biom_path)

In [1099]:
V1V3_tbl

,g__Cutibacterium,g__uncultured,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Veillonella,g__Micrococcus,g__Alloprevotella,g__Lactobacillus,...,g__Ramlibacter,g__Delftia,g__Lacibacter,g__Luteitalea,g__Pseudochrobactrum,g__Methylotenera,g__Fusicatenibacter,g__Ruminococcus,g__Lachnospira,g__Parabacteroides
LAMI.RD001.D0.C1,2362.0,0.0,94.0,292.0,175.0,91.0,33.0,37.0,18.0,1.0,...,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D0.C2,1808.0,2.0,158.0,374.0,352.0,120.0,36.0,52.0,27.0,165.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D14.C1,2234.0,2.0,83.0,240.0,492.0,253.0,19.0,30.0,30.0,60.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D14.C2,1761.0,0.0,137.0,446.0,456.0,151.0,69.0,16.0,100.0,26.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D28.C1,2367.0,11.0,118.0,293.0,365.0,217.0,34.0,22.0,38.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D14.C1,1900.0,1846.0,8.0,0.0,6.0,8.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D21.C3,1003.0,2741.0,11.0,1.0,4.0,9.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C2,2315.0,1410.0,8.0,0.0,9.0,27.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D9.C1,1230.0,2491.0,13.0,1.0,3.0,31.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [1100]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_V1V3_tbl = metaB_tbl[metaB_tbl.index.isin(V1V3_tbl.index)]

# Remove the 'group' column (only for VIP table)
# metaB_V1V3_tbl = metaB_V1V3_tbl.drop(columns=['group'])
metaB_V1V3_tbl.index.name = None
metaB_V1V3_tbl

,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
LAMI.RD001.D0.C1,2987383.80,2025607.40,29574.6480,39859.7580,28277.594,0.00,970487.500,1942604.500,567510.060,1473068.10,...,62007.1100,12406.8260,611709.700,16778.8380,0.0000,0.000,0.000,0.00,0.000,125692.330
LAMI.RD306.D9.C2,2051791.50,2260333.20,34466.4840,0.0000,253083.810,178911.47,1071868.200,3572847.500,672054.400,308019.97,...,153381.2700,20433.4900,437938.200,12865.3900,0.0000,0.000,0.000,0.00,0.000,10553.927
LAMI.RD304.D11.C1,1818308.60,1434033.20,47379.2730,36750.4000,15391.807,0.00,595568.940,1605551.000,328151.120,299140.06,...,58588.9260,9403.6520,78491.530,0.0000,5375.8525,11225.194,23546.951,99470.69,29824.627,12473.092
LAMI.RD304.D11.C2,1126503.20,706197.75,3460.0974,0.0000,15015.189,0.00,303845.300,865753.500,152392.720,191713.67,...,20153.3320,2286.2397,261996.800,15780.3090,2691.0256,16378.752,34863.656,157503.92,44084.586,40673.110
LAMI.RD304.D0.C1,827466.30,778566.10,4547.2640,3345.8184,45187.030,0.00,293386.340,1321474.400,137201.390,705570.80,...,45193.9400,0.0000,427202.060,3228.1143,3431.0034,0.000,0.000,0.00,0.000,49464.902
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,1173415.00,1250164.60,8613.8590,9787.1210,14712.559,0.00,579058.800,1380958.500,498893.780,236111.75,...,24563.9240,5144.6616,122958.260,56654.0430,0.0000,0.000,0.000,0.00,0.000,0.000
LAMI.RD308.D9.C3,550094.94,687527.06,3163.9185,3076.9688,17934.553,0.00,281896.700,742416.600,262886.470,303509.40,...,24480.3900,0.0000,65659.700,21826.1170,0.0000,0.000,0.000,0.00,0.000,220846.160
LAMI.RD319.D2.C2,602147.10,538412.94,4006.6484,0.0000,10556.326,0.00,255542.720,1194061.900,121803.890,5406879.00,...,35224.5800,0.0000,73231.560,19741.2460,0.0000,0.000,0.000,0.00,0.000,25695.014
LAMI.RD319.D2.C3,171846.72,106901.95,0.0000,1407.5230,25369.129,0.00,59064.336,833009.000,23485.404,2838344.20,...,25342.2100,0.0000,55986.290,0.0000,0.0000,0.000,0.000,0.00,0.000,11039.730


In [1101]:
# Convert to relative abundance (row-wise normalization)
metaB_V1V3_tbl_relative = metaB_V1V3_tbl.div(metaB_V1V3_tbl.sum(axis=1), axis=0)
metaB_V1V3_tbl_relative

,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
LAMI.RD001.D0.C1,0.265648,0.180123,0.002630,0.003544,0.002515,0.000000,0.086299,0.172743,0.050465,0.130990,...,0.005514,0.001103,0.054395,0.001492,0.000000,0.000000,0.000000,0.000000,0.000000,0.011177
LAMI.RD306.D9.C2,0.177092,0.195092,0.002975,0.000000,0.021844,0.015442,0.092514,0.308376,0.058006,0.026585,...,0.013238,0.001764,0.037799,0.001110,0.000000,0.000000,0.000000,0.000000,0.000000,0.000911
LAMI.RD304.D11.C1,0.269227,0.212329,0.007015,0.005441,0.002279,0.000000,0.088183,0.237725,0.048588,0.044292,...,0.008675,0.001392,0.011622,0.000000,0.000796,0.001662,0.003486,0.014728,0.004416,0.001847
LAMI.RD304.D11.C2,0.274761,0.172246,0.000844,0.000000,0.003662,0.000000,0.074110,0.211162,0.037169,0.046760,...,0.004916,0.000558,0.063903,0.003849,0.000656,0.003995,0.008503,0.038416,0.010752,0.009920
LAMI.RD304.D0.C1,0.172611,0.162410,0.000949,0.000698,0.009426,0.000000,0.061201,0.275662,0.028620,0.147184,...,0.009428,0.000000,0.089115,0.000673,0.000716,0.000000,0.000000,0.000000,0.000000,0.010318
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.210388,0.224149,0.001544,0.001755,0.002638,0.000000,0.103823,0.247600,0.089449,0.042334,...,0.004404,0.000922,0.022046,0.010158,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD308.D9.C3,0.139416,0.174247,0.000802,0.000780,0.004545,0.000000,0.071444,0.188158,0.066626,0.076921,...,0.006204,0.000000,0.016641,0.005532,0.000000,0.000000,0.000000,0.000000,0.000000,0.055971
LAMI.RD319.D2.C2,0.071895,0.064285,0.000478,0.000000,0.001260,0.000000,0.030511,0.142568,0.014543,0.645566,...,0.004206,0.000000,0.008744,0.002357,0.000000,0.000000,0.000000,0.000000,0.000000,0.003068
LAMI.RD319.D2.C3,0.040939,0.025467,0.000000,0.000335,0.006044,0.000000,0.014071,0.198447,0.005595,0.676175,...,0.006037,0.000000,0.013338,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002630


In [1102]:
# Filter metadata for the correct group
filtered_metadata = metadata[metadata['group'] == skin_group]

# Normalize SampleID and index formatting
metaB_V1V3_tbl_relative.index = metaB_V1V3_tbl_relative.index.str.strip().str.upper()
filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()

# Filter metadata to only include matching SampleIDs
filtered_metadata = filtered_metadata[filtered_metadata['SampleID'].isin(metaB_V1V3_tbl_relative.index)]

# Subset the DataFrame
metaB_V1V3_tbl_relative = metaB_V1V3_tbl_relative.loc[filtered_metadata['SampleID']]

metaB_V1V3_tbl_relative

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_3481/3342871550.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()


,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
LAMI.RD002.D14.C1,0.177464,0.159415,0.002228,0.000646,0.010071,0.000000,0.081174,0.334196,0.034301,0.069704,...,0.016729,0.000396,0.060050,0.001087,0.0,0.000000,0.008208,0.000000,0.019387,0.000000
LAMI.RD003.D14.C1,0.043985,0.036410,0.000000,0.000000,0.004719,0.000000,0.014899,0.085250,0.006927,0.790429,...,0.004982,0.000000,0.001511,0.000286,0.0,0.000000,0.000000,0.000000,0.000000,0.001695
LAMI.RD017.D0.C2,0.225856,0.195690,0.000000,0.000997,0.001983,0.002688,0.096634,0.287196,0.056365,0.033499,...,0.008864,0.000000,0.037747,0.003979,0.0,0.000000,0.000000,0.003933,0.000000,0.000581
LAMI.RD007.D14.C1,0.206929,0.171066,0.001356,0.000000,0.005920,0.010829,0.076215,0.302280,0.027798,0.095579,...,0.008078,0.000993,0.054766,0.003050,0.0,0.000000,0.000000,0.000000,0.000000,0.017369
LAMI.RD017.D28.C2,0.241377,0.257380,0.000284,0.000256,0.000982,0.010516,0.127992,0.205061,0.077467,0.012105,...,0.005975,0.002195,0.024483,0.003850,0.0,0.000000,0.000000,0.001510,0.000000,0.000335
LAMI.RD002.D28.C2,0.045730,0.024322,0.000070,0.000000,0.002893,0.001367,0.010188,0.086950,0.006321,0.810122,...,0.002050,0.000034,0.000179,0.001058,0.0,0.000000,0.000000,0.001187,0.000000,0.004675
LAMI.RD004.D0.C1,0.177376,0.094539,0.000904,0.001157,0.003262,0.019609,0.038629,0.277297,0.028480,0.262060,...,0.008955,0.000168,0.058771,0.000000,0.0,0.000000,0.000000,0.004322,0.000000,0.000000
LAMI.RD003.D28.C1,0.210181,0.108887,0.000000,0.000752,0.003702,0.000000,0.043909,0.165153,0.029659,0.380396,...,0.006914,0.000358,0.003779,0.003913,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD001.D14.C1,0.230963,0.232861,0.003391,0.002454,0.001484,0.000888,0.105728,0.195136,0.049886,0.035618,...,0.006185,0.003094,0.080559,0.000359,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD001.D28.C1,0.295443,0.196816,0.002237,0.003003,0.003460,0.000270,0.104208,0.208874,0.075193,0.021648,...,0.007380,0.005108,0.015558,0.001258,0.0,0.000000,0.000000,0.002090,0.000000,0.002443


In [1103]:
# Save as biom for mmvec
metaB_V1V3_tbl_transposed = metaB_V1V3_tbl_relative.T
output_path = f'../Data/multi-omics/metabolomics-tbl_16S_V1V3-matched_relative_{skin_group}.biom'
save_as_biom(metaB_V1V3_tbl_relative, output_path)

In [1104]:
# Subset V1V3_tbl rows where sample is in metaB_tbl
V1V3_tbl_matched = V1V3_tbl[V1V3_tbl.index.isin(metaB_V1V3_tbl_relative.index)]

top_V1V3_features = [' g__Cutibacterium', ' g__Staphylococcus',
                     ' g__Streptococcus',' g__Lawsonella',
                     ' g__Micrococcus', ' g__Lactobacillus', 
                     ' g__Rothia', ' g__Kocuria', ' g__Haemophilus', ' g__Corynebacterium', ' g__Veillonella']

# top_V1V3_features = ['g__Cutibacterium|ASV1', 'g__Staphylococcus|ASV1',
#                      'g__Streptococcus|ASV1','g__Lawsonella',
#                      ' g__Micrococcus|ASV1', 'g__Lactobacillus|ASV1', 
#                      'g__Rothia|ASV1', 'g__Kocuria|ASV1', 'g__Haemophilus|ASV1', 'g__Corynebacterium|ASV1']

# Find the intersection of desired columns and actual DataFrame columns
available_columns = V1V3_tbl_matched.columns.intersection(top_V1V3_features)
V1V3_tbl_matched = V1V3_tbl_matched[available_columns]

V1V3_tbl_matched

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Veillonella,g__Micrococcus,g__Lactobacillus,g__Rothia,g__Haemophilus,g__Kocuria
LAMI.RD001.D0.C1,2362.0,94.0,292.0,175.0,91.0,33.0,37.0,1.0,50.0,87.0,0.0
LAMI.RD001.D0.C2,1808.0,158.0,374.0,352.0,120.0,36.0,52.0,165.0,43.0,74.0,9.0
LAMI.RD001.D14.C1,2234.0,83.0,240.0,492.0,253.0,19.0,30.0,60.0,12.0,8.0,0.0
LAMI.RD001.D14.C2,1761.0,137.0,446.0,456.0,151.0,69.0,16.0,26.0,31.0,58.0,6.0
LAMI.RD001.D28.C1,2367.0,118.0,293.0,365.0,217.0,34.0,22.0,17.0,14.0,10.0,5.0
LAMI.RD002.D0.C2,2900.0,373.0,159.0,39.0,13.0,2.0,61.0,1.0,27.0,11.0,54.0
LAMI.RD003.D14.C1,3296.0,95.0,14.0,138.0,14.0,0.0,0.0,1.0,5.0,0.0,0.0
LAMI.RD002.D14.C1,3440.0,105.0,43.0,49.0,41.0,0.0,7.0,2.0,4.0,1.0,0.0
LAMI.RD003.D28.C1,2367.0,103.0,543.0,31.0,36.0,0.0,0.0,0.0,16.0,21.0,0.0
LAMI.RD001.D28.C2,2288.0,204.0,169.0,399.0,420.0,25.0,20.0,11.0,20.0,11.0,9.0


In [1105]:
# Convert to relative abundance (row-wise normalization)
V1V3_tbl_matched_relative = V1V3_tbl_matched.div(V1V3_tbl_matched.sum(axis=1), axis=0)
V1V3_tbl_matched_relative

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Veillonella,g__Micrococcus,g__Lactobacillus,g__Rothia,g__Haemophilus,g__Kocuria
LAMI.RD001.D0.C1,0.733085,0.029174,0.090627,0.054314,0.028243,0.010242,0.011484,0.000310,0.015518,0.027002,0.000000
LAMI.RD001.D0.C2,0.566594,0.049514,0.117205,0.110310,0.037606,0.011282,0.016296,0.051708,0.013475,0.023190,0.002820
LAMI.RD001.D14.C1,0.651122,0.024191,0.069950,0.143398,0.073739,0.005538,0.008744,0.017488,0.003498,0.002332,0.000000
LAMI.RD001.D14.C2,0.557808,0.043396,0.141273,0.144441,0.047830,0.021856,0.005068,0.008236,0.009819,0.018372,0.001901
LAMI.RD001.D28.C1,0.683709,0.034084,0.084633,0.105430,0.062681,0.009821,0.006355,0.004910,0.004044,0.002889,0.001444
LAMI.RD002.D0.C2,0.796703,0.102473,0.043681,0.010714,0.003571,0.000549,0.016758,0.000275,0.007418,0.003022,0.014835
LAMI.RD003.D14.C1,0.925063,0.026663,0.003929,0.038731,0.003929,0.000000,0.000000,0.000281,0.001403,0.000000,0.000000
LAMI.RD002.D14.C1,0.931744,0.028440,0.011647,0.013272,0.011105,0.000000,0.001896,0.000542,0.001083,0.000271,0.000000
LAMI.RD003.D28.C1,0.759384,0.033045,0.174206,0.009945,0.011550,0.000000,0.000000,0.000000,0.005133,0.006737,0.000000
LAMI.RD001.D28.C2,0.639821,0.057047,0.047260,0.111577,0.117450,0.006991,0.005593,0.003076,0.005593,0.003076,0.002517


In [1106]:
# Save metadata for V1V3
metadata_V1V3 = metadata[metadata['SampleID'].isin(V1V3_tbl_matched_relative.index)]
metadata_V1V3 = metadata_V1V3[['SampleID', 'group']]

# Rename the first column to 'sample id'
metadata_V1V3.rename(columns={metadata_V1V3.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V1V3.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V1V3.columns[1:])]

metadata_V1V3.to_csv('../Data/multi-omics/16S_V1V3_metaB-matched.csv', index=0)
metadata_V1V3

,sample id,group
9,LAMI.RD002.D14.C1,Healthy
14,LAMI.RD003.D14.C1,Healthy
22,LAMI.RD017.D0.C2,Healthy
27,LAMI.RD007.D14.C1,Healthy
28,LAMI.RD017.D28.C2,Healthy
30,LAMI.RD002.D28.C2,Healthy
33,LAMI.RD004.D0.C1,Healthy
36,LAMI.RD003.D28.C1,Healthy
38,LAMI.RD001.D14.C1,Healthy
45,LAMI.RD001.D28.C1,Healthy


In [1107]:
# Save as biom for mmvec
V1V3_tbl_matched_transposed = V1V3_tbl_matched_relative.T
output_path = f'../Data/multi-omics/16S_V1V3-tbl_metaB-matched_relative_{skin_group}.biom'
save_as_biom(V1V3_tbl_matched_transposed, output_path)

In [1108]:
# Create column_names_df with 'feature id' as the column header
column_names_df = pd.DataFrame({"feature id": metaB_V1V3_tbl.columns})  # Ensure feature id is a string

column_names_df

,feature id
0,"Pyroglutamic acid (mz: 130.0497016, RT: 0.6479..."
1,"(Iso)leucine (mz: 132.1017009, RT: 0.71511286)"
2,"Urocanic acid (mz: 139.0410336, RT: 0.6406374)"
3,"Urocanic acid (mz: 139.0587028, RT: 0.6315576)"
4,"Paracetamol (mz: 152.0704373, RT: 1.6221956)"
5,"Nicotine (mz: 163.1224157, RT: 0.634317)"
6,"Phenylalanine (mz: 166.086109, RT: 0.9777797)"
7,"Tyrosine (mz: 182.0809391, RT: 0.6720225)"
8,"Tryptophan (mz: 205.0969182, RT: 2.0760007)"
9,"DL-Panthenol (mz: 206.1384472, RT: 1.1194918)"


In [1109]:
# Save to a CSV file
output_path = "../Data/multi-omics/metabolite_info.csv"
column_names_df = column_names_df.to_csv(output_path, index=0)

In [1110]:
# Subset metaB rows where sample is in V4_tbl
metaB_V4_tbl = metaB_tbl[metaB_tbl.index.isin(V4_tbl.index)]
# metaB_V4_tbl = metaB_V4_tbl.drop('group', axis=1)
metaB_V4_tbl.index.name = None
metaB_V4_tbl

,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
LAMI.RD001.D0.C1,2987383.80,2025607.40,29574.6480,39859.7580,28277.594,0.00,970487.500,1942604.500,567510.060,1473068.10,...,62007.1100,12406.8260,611709.700,16778.8380,0.0000,0.000,0.000,0.00,0.000,125692.330
LAMI.RD306.D9.C2,2051791.50,2260333.20,34466.4840,0.0000,253083.810,178911.47,1071868.200,3572847.500,672054.400,308019.97,...,153381.2700,20433.4900,437938.200,12865.3900,0.0000,0.000,0.000,0.00,0.000,10553.927
LAMI.RD304.D11.C1,1818308.60,1434033.20,47379.2730,36750.4000,15391.807,0.00,595568.940,1605551.000,328151.120,299140.06,...,58588.9260,9403.6520,78491.530,0.0000,5375.8525,11225.194,23546.951,99470.69,29824.627,12473.092
LAMI.RD304.D11.C2,1126503.20,706197.75,3460.0974,0.0000,15015.189,0.00,303845.300,865753.500,152392.720,191713.67,...,20153.3320,2286.2397,261996.800,15780.3090,2691.0256,16378.752,34863.656,157503.92,44084.586,40673.110
LAMI.RD304.D0.C1,827466.30,778566.10,4547.2640,3345.8184,45187.030,0.00,293386.340,1321474.400,137201.390,705570.80,...,45193.9400,0.0000,427202.060,3228.1143,3431.0034,0.000,0.000,0.00,0.000,49464.902
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,1173415.00,1250164.60,8613.8590,9787.1210,14712.559,0.00,579058.800,1380958.500,498893.780,236111.75,...,24563.9240,5144.6616,122958.260,56654.0430,0.0000,0.000,0.000,0.00,0.000,0.000
LAMI.RD308.D9.C3,550094.94,687527.06,3163.9185,3076.9688,17934.553,0.00,281896.700,742416.600,262886.470,303509.40,...,24480.3900,0.0000,65659.700,21826.1170,0.0000,0.000,0.000,0.00,0.000,220846.160
LAMI.RD319.D2.C2,602147.10,538412.94,4006.6484,0.0000,10556.326,0.00,255542.720,1194061.900,121803.890,5406879.00,...,35224.5800,0.0000,73231.560,19741.2460,0.0000,0.000,0.000,0.00,0.000,25695.014
LAMI.RD319.D2.C3,171846.72,106901.95,0.0000,1407.5230,25369.129,0.00,59064.336,833009.000,23485.404,2838344.20,...,25342.2100,0.0000,55986.290,0.0000,0.0000,0.000,0.000,0.00,0.000,11039.730


In [1111]:
# Drop the column from the DataFrame as it wasn't significant after univariate testing
# metaB_V4_tbl = metaB_V4_tbl.drop(columns=['C19H22O4 Linear diarylheptanoids'])

In [1112]:
# Convert to relative abundance (row-wise normalization)
metaB_V4_tbl_relative = metaB_V4_tbl.div(metaB_V4_tbl.sum(axis=1), axis=0)
metaB_V4_tbl_relative

,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
LAMI.RD001.D0.C1,0.265648,0.180123,0.002630,0.003544,0.002515,0.000000,0.086299,0.172743,0.050465,0.130990,...,0.005514,0.001103,0.054395,0.001492,0.000000,0.000000,0.000000,0.000000,0.000000,0.011177
LAMI.RD306.D9.C2,0.177092,0.195092,0.002975,0.000000,0.021844,0.015442,0.092514,0.308376,0.058006,0.026585,...,0.013238,0.001764,0.037799,0.001110,0.000000,0.000000,0.000000,0.000000,0.000000,0.000911
LAMI.RD304.D11.C1,0.269227,0.212329,0.007015,0.005441,0.002279,0.000000,0.088183,0.237725,0.048588,0.044292,...,0.008675,0.001392,0.011622,0.000000,0.000796,0.001662,0.003486,0.014728,0.004416,0.001847
LAMI.RD304.D11.C2,0.274761,0.172246,0.000844,0.000000,0.003662,0.000000,0.074110,0.211162,0.037169,0.046760,...,0.004916,0.000558,0.063903,0.003849,0.000656,0.003995,0.008503,0.038416,0.010752,0.009920
LAMI.RD304.D0.C1,0.172611,0.162410,0.000949,0.000698,0.009426,0.000000,0.061201,0.275662,0.028620,0.147184,...,0.009428,0.000000,0.089115,0.000673,0.000716,0.000000,0.000000,0.000000,0.000000,0.010318
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.210388,0.224149,0.001544,0.001755,0.002638,0.000000,0.103823,0.247600,0.089449,0.042334,...,0.004404,0.000922,0.022046,0.010158,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD308.D9.C3,0.139416,0.174247,0.000802,0.000780,0.004545,0.000000,0.071444,0.188158,0.066626,0.076921,...,0.006204,0.000000,0.016641,0.005532,0.000000,0.000000,0.000000,0.000000,0.000000,0.055971
LAMI.RD319.D2.C2,0.071895,0.064285,0.000478,0.000000,0.001260,0.000000,0.030511,0.142568,0.014543,0.645566,...,0.004206,0.000000,0.008744,0.002357,0.000000,0.000000,0.000000,0.000000,0.000000,0.003068
LAMI.RD319.D2.C3,0.040939,0.025467,0.000000,0.000335,0.006044,0.000000,0.014071,0.198447,0.005595,0.676175,...,0.006037,0.000000,0.013338,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002630


In [1113]:
# Filter metadata for the correct group
filtered_metadata = metadata[metadata['group'] == skin_group]

# Normalize SampleID and index formatting
metaB_V4_tbl_relative.index = metaB_V4_tbl_relative.index.str.strip().str.upper()
filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()

# Filter metadata to only include matching SampleIDs
filtered_metadata = filtered_metadata[filtered_metadata['SampleID'].isin(metaB_V4_tbl_relative.index)]

# Subset the DataFrame
metaB_V4_tbl_relative = metaB_V4_tbl_relative.loc[filtered_metadata['SampleID']]

metaB_V4_tbl_relative

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_3481/4148729606.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()


,"Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Paracetamol (mz: 152.0704373, RT: 1.6221956)","Nicotine (mz: 163.1224157, RT: 0.634317)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","DL-Panthenol (mz: 206.1384472, RT: 1.1194918)",...,"(Iso)leucine (mz: 276.143683, RT: 0.6789317)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","Nobiletin (mz: 403.138254, RT: 5.258404)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)"
LAMI.RD017.D0.C1,0.267509,0.198181,0.000000,0.000799,0.002173,0.001165,0.105062,0.223399,0.069073,0.044435,...,0.007956,0.000000,0.031120,0.003507,0.000445,0.0,0.000000,0.004438,0.0,0.000000
LAMI.RD017.D0.C2,0.225856,0.195690,0.000000,0.000997,0.001983,0.002688,0.096634,0.287196,0.056365,0.033499,...,0.008864,0.000000,0.037747,0.003979,0.000000,0.0,0.000000,0.003933,0.0,0.000581
LAMI.RD007.D14.C1,0.206929,0.171066,0.001356,0.000000,0.005920,0.010829,0.076215,0.302280,0.027798,0.095579,...,0.008078,0.000993,0.054766,0.003050,0.000000,0.0,0.000000,0.000000,0.0,0.017369
LAMI.RD017.D28.C2,0.241377,0.257380,0.000284,0.000256,0.000982,0.010516,0.127992,0.205061,0.077467,0.012105,...,0.005975,0.002195,0.024483,0.003850,0.000000,0.0,0.000000,0.001510,0.0,0.000335
LAMI.RD004.D0.C1,0.177376,0.094539,0.000904,0.001157,0.003262,0.019609,0.038629,0.277297,0.028480,0.262060,...,0.008955,0.000168,0.058771,0.000000,0.000000,0.0,0.000000,0.004322,0.0,0.000000
LAMI.RD001.D14.C1,0.230963,0.232861,0.003391,0.002454,0.001484,0.000888,0.105728,0.195136,0.049886,0.035618,...,0.006185,0.003094,0.080559,0.000359,0.000000,0.0,0.000000,0.000000,0.0,0.000000
LAMI.RD001.D28.C1,0.295443,0.196816,0.002237,0.003003,0.003460,0.000270,0.104208,0.208874,0.075193,0.021648,...,0.007380,0.005108,0.015558,0.001258,0.000000,0.0,0.000000,0.002090,0.0,0.002443
LAMI.RD017.D28.C1,0.220370,0.251676,0.002644,0.000304,0.001835,0.011191,0.117802,0.229543,0.074486,0.022402,...,0.010360,0.001639,0.026570,0.003333,0.000000,0.0,0.000000,0.001841,0.0,0.000000
LAMI.RD017.D14.C2,0.266732,0.226188,0.001128,0.000513,0.001460,0.002557,0.097817,0.189710,0.055908,0.097200,...,0.003415,0.001375,0.026563,0.001489,0.000274,0.0,0.000000,0.000730,0.0,0.002643
LAMI.RD006.D14.C1,0.087079,0.037841,0.000000,0.000344,0.000427,0.008884,0.014522,0.052685,0.008457,0.779160,...,0.001562,0.000000,0.000313,0.001936,0.000000,0.0,0.000000,0.000000,0.0,0.000541


In [1114]:
# Subset V4_tbl rows where sample is in metaB_tbl
V4_tbl_matched = V4_tbl[V4_tbl.index.isin(metaB_V4_tbl_relative.index)]

# top_V4_features = [' g__Staphylococcus', ' g__Lawsonella',
#                     ' g__Streptococcus', ' g__Acinetobacter', ' g__Cutibacterium', 'g__Enhydrobacter',
#                      ' g__Micrococcus', ' g__Lactobacillus', ' g__Rothia', ' g__Kocuria', ' g__Haemophilus', ' g__Corynebacterium']


top_V4_features = [' g__Staphylococcus', ' g__Lawsonella', ' g__Streptococcus', ' g__Cutibacterium',
                     ' g__Micrococcus', ' g__Lactobacillus', ' g__Rothia', ' g__Kocuria', ' g__Haemophilus', ' g__Corynebacterium', ' g__Veillonella']

# Find the intersection of desired columns and actual DataFrame columns
available_columns = V4_tbl_matched.columns.intersection(top_V4_features)
V4_tbl_matched = V4_tbl_matched[available_columns]

V4_tbl_matched

,g__Staphylococcus,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Veillonella,g__Cutibacterium,g__Kocuria,g__Rothia
LAMI.RD001.D0.C1,317.0,131.0,275.0,414.0,475.0,100.0,15.0,43.0,1.0,17.0,36.0
LAMI.RD001.D14.C1,454.0,456.0,915.0,247.0,168.0,93.0,117.0,41.0,0.0,5.0,15.0
LAMI.RD004.D0.C2,1652.0,95.0,848.0,164.0,28.0,81.0,97.0,59.0,1.0,18.0,49.0
LAMI.RD001.D0.C2,349.0,217.0,509.0,368.0,464.0,75.0,182.0,37.0,3.0,3.0,56.0
LAMI.RD011.D0.C2,2132.0,291.0,220.0,153.0,227.0,60.0,26.0,32.0,3.0,44.0,27.0
LAMI.RD001.D14.C2,444.0,206.0,546.0,457.0,456.0,54.0,23.0,83.0,0.0,5.0,47.0
LAMI.RD004.D28.C2,217.0,18.0,731.0,713.0,44.0,276.0,40.0,115.0,2.0,16.0,15.0
LAMI.RD017.D14.C2,1756.0,636.0,131.0,33.0,25.0,617.0,2.0,0.0,126.0,7.0,1.0
LAMI.RD011.D14.C1,2063.0,999.0,102.0,85.0,37.0,88.0,45.0,15.0,32.0,3.0,7.0
LAMI.RD004.D0.C1,603.0,48.0,875.0,270.0,14.0,83.0,21.0,21.0,6.0,5.0,30.0


In [1115]:
# Convert to relative abundance (row-wise normalization)
V4_tbl_matched_relative = V4_tbl_matched.div(V4_tbl_matched.sum(axis=1), axis=0)
V4_tbl_matched_relative

,g__Staphylococcus,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Veillonella,g__Cutibacterium,g__Kocuria,g__Rothia
LAMI.RD001.D0.C1,0.173794,0.071820,0.150768,0.226974,0.260417,0.054825,0.008224,0.023575,0.000548,0.009320,0.019737
LAMI.RD001.D14.C1,0.180804,0.181601,0.364397,0.098367,0.066906,0.037037,0.046595,0.016328,0.000000,0.001991,0.005974
LAMI.RD004.D0.C2,0.534282,0.030724,0.274256,0.053040,0.009056,0.026197,0.031371,0.019082,0.000323,0.005821,0.015847
LAMI.RD001.D0.C2,0.154220,0.095890,0.224923,0.162616,0.205038,0.033142,0.080424,0.016350,0.001326,0.001326,0.024746
LAMI.RD011.D0.C2,0.663142,0.090513,0.068429,0.047589,0.070607,0.018663,0.008087,0.009953,0.000933,0.013686,0.008398
LAMI.RD001.D14.C2,0.191297,0.088755,0.235243,0.196898,0.196467,0.023266,0.009910,0.035760,0.000000,0.002154,0.020250
LAMI.RD004.D28.C2,0.099223,0.008230,0.334248,0.326017,0.020119,0.126200,0.018290,0.052583,0.000914,0.007316,0.006859
LAMI.RD017.D14.C2,0.526695,0.190762,0.039292,0.009898,0.007499,0.185063,0.000600,0.000000,0.037792,0.002100,0.000300
LAMI.RD011.D14.C1,0.593498,0.287399,0.029344,0.024453,0.010644,0.025316,0.012946,0.004315,0.009206,0.000863,0.002014
LAMI.RD004.D0.C1,0.305162,0.024291,0.442814,0.136640,0.007085,0.042004,0.010628,0.010628,0.003036,0.002530,0.015182


In [1116]:
# Save metadata for V4
metadata_V4 = metadata[metadata['SampleID'].isin(V4_tbl_matched.index)]
metadata_V4 = metadata_V4[['SampleID', 'group']]
# Rename the first column to 'sample id'
metadata_V4.rename(columns={metadata_V4.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V4.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V4.columns[1:])]

metadata_V4.to_csv('../Data/multi-omics/16S_V4_metaB-matched.csv', index=0)
metadata_V4

,sample id,group
18,LAMI.RD017.D0.C1,Healthy
22,LAMI.RD017.D0.C2,Healthy
27,LAMI.RD007.D14.C1,Healthy
28,LAMI.RD017.D28.C2,Healthy
33,LAMI.RD004.D0.C1,Healthy
38,LAMI.RD001.D14.C1,Healthy
45,LAMI.RD001.D28.C1,Healthy
46,LAMI.RD017.D28.C1,Healthy
49,LAMI.RD017.D14.C2,Healthy
58,LAMI.RD006.D14.C1,Healthy


In [1117]:
# Save as biom for mmvec
metaB_V4_tbl_transposed = metaB_V4_tbl_relative.T
output_path = f'../Data/multi-omics/metabolomics-tbl_16S_V4-matched_relative_{skin_group}.biom'
save_as_biom(metaB_V4_tbl_transposed, output_path)

In [1118]:
# Save as biom for mmvec
V4_tbl_matched_transposed = V4_tbl_matched_relative.T
output_path = '../Data/multi-omics/16S_V4-tbl_metaB-matched_relative.biom'
save_as_biom(V4_tbl_matched_transposed, output_path)

### Combine V1-V3 and V4 tables

In [1119]:
# Find common rows (samples)
common_rows = V1V3_tbl_matched.index.intersection(V4_tbl_matched.index)

# Subset both DataFrames to keep only common rows
V1V3_tbl_matched = V1V3_tbl_matched.loc[common_rows]
V4_tbl_matched = V4_tbl_matched.loc[common_rows]

# Merge by taking average of values row-wise
merged_tbl = V1V3_tbl_matched.add(V4_tbl_matched, fill_value=0)

# Sort columns by total sum (descending order)
merged_tbl = merged_tbl[merged_tbl.sum().sort_values(ascending=False).index]

merged_tbl

,g__Cutibacterium,g__Staphylococcus,g__Corynebacterium,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Haemophilus,g__Kocuria,g__Lactobacillus,g__Veillonella,g__Rothia
LAMI.RD001.D0.C1,2363.0,411.0,450.0,706.0,222.0,137.0,562.0,17.0,16.0,76.0,86.0
LAMI.RD001.D0.C2,1811.0,507.0,861.0,742.0,337.0,127.0,538.0,12.0,347.0,73.0,99.0
LAMI.RD001.D14.C1,2234.0,537.0,1407.0,487.0,709.0,123.0,176.0,5.0,177.0,60.0,27.0
LAMI.RD001.D14.C2,1761.0,581.0,1002.0,903.0,357.0,70.0,514.0,11.0,49.0,152.0,78.0
LAMI.RD001.D28.C1,2368.0,698.0,1057.0,642.0,674.0,94.0,152.0,10.0,100.0,84.0,48.0
LAMI.RD001.D28.C2,2292.0,1053.0,1261.0,448.0,913.0,97.0,125.0,14.0,41.0,83.0,45.0
LAMI.RD007.D14.C1,3647.0,2094.0,297.0,183.0,367.0,40.0,78.0,0.0,7.0,74.0,31.0
LAMI.RD007.D14.C2,3598.0,1567.0,379.0,168.0,507.0,63.0,141.0,0.0,0.0,65.0,0.0
LAMI.RD004.D0.C1,2680.0,705.0,1329.0,580.0,70.0,104.0,17.0,5.0,32.0,60.0,59.0
LAMI.RD006.D14.C2,3400.0,774.0,917.0,131.0,416.0,156.0,15.0,33.0,132.0,42.0,3.0


In [1120]:
V1V3_tbl_matched.columns

Index([' g__Cutibacterium', ' g__Staphylococcus', ' g__Streptococcus',
       ' g__Corynebacterium', ' g__Lawsonella', ' g__Veillonella',
       ' g__Micrococcus', ' g__Lactobacillus', ' g__Rothia', ' g__Haemophilus',
       ' g__Kocuria'],
      dtype='object')

In [1121]:
V4_tbl_matched.columns

Index([' g__Staphylococcus', ' g__Lawsonella', ' g__Corynebacterium',
       ' g__Streptococcus', ' g__Haemophilus', ' g__Micrococcus',
       ' g__Lactobacillus', ' g__Veillonella', ' g__Cutibacterium',
       ' g__Kocuria', ' g__Rothia'],
      dtype='object')

In [1122]:
# Find common rows (samples)
common_rows = V1V3_tbl_matched.index.intersection(V4_tbl_matched.index)

# Subset both DataFrames to keep only common rows
V1V3_tbl_matched = V1V3_tbl_matched.loc[common_rows]
V4_tbl_matched = V4_tbl_matched.loc[common_rows]

# Compute the mean for each matching row and column
merged_tbl = (V1V3_tbl_matched + V4_tbl_matched) / 2

# Sort columns by total mean (descending order)
merged_tbl = merged_tbl[merged_tbl.mean().sort_values(ascending=False).index]

# Display merged DataFrame
merged_tbl


,g__Cutibacterium,g__Staphylococcus,g__Corynebacterium,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Haemophilus,g__Kocuria,g__Lactobacillus,g__Veillonella,g__Rothia
LAMI.RD001.D0.C1,1181.5,205.5,225.0,353.0,111.0,68.5,281.0,8.5,8.0,38.0,43.0
LAMI.RD001.D0.C2,905.5,253.5,430.5,371.0,168.5,63.5,269.0,6.0,173.5,36.5,49.5
LAMI.RD001.D14.C1,1117.0,268.5,703.5,243.5,354.5,61.5,88.0,2.5,88.5,30.0,13.5
LAMI.RD001.D14.C2,880.5,290.5,501.0,451.5,178.5,35.0,257.0,5.5,24.5,76.0,39.0
LAMI.RD001.D28.C1,1184.0,349.0,528.5,321.0,337.0,47.0,76.0,5.0,50.0,42.0,24.0
LAMI.RD001.D28.C2,1146.0,526.5,630.5,224.0,456.5,48.5,62.5,7.0,20.5,41.5,22.5
LAMI.RD007.D14.C1,1823.5,1047.0,148.5,91.5,183.5,20.0,39.0,0.0,3.5,37.0,15.5
LAMI.RD007.D14.C2,1799.0,783.5,189.5,84.0,253.5,31.5,70.5,0.0,0.0,32.5,0.0
LAMI.RD004.D0.C1,1340.0,352.5,664.5,290.0,35.0,52.0,8.5,2.5,16.0,30.0,29.5
LAMI.RD006.D14.C2,1700.0,387.0,458.5,65.5,208.0,78.0,7.5,16.5,66.0,21.0,1.5


In [1123]:
# Convert to relative abundance (row-wise normalization)
merged_tbl_relative = merged_tbl.div(merged_tbl.sum(axis=1), axis=0)
merged_tbl_relative

,g__Cutibacterium,g__Staphylococcus,g__Corynebacterium,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Haemophilus,g__Kocuria,g__Lactobacillus,g__Veillonella,g__Rothia
LAMI.RD001.D0.C1,0.468292,0.081451,0.089180,0.139913,0.043995,0.027150,0.111375,0.003369,0.003171,0.015061,0.017043
LAMI.RD001.D0.C2,0.332050,0.092959,0.157866,0.136047,0.061790,0.023286,0.098643,0.002200,0.063623,0.013385,0.018152
LAMI.RD001.D14.C1,0.375968,0.090374,0.236789,0.081959,0.119320,0.020700,0.029620,0.000841,0.029788,0.010098,0.004544
LAMI.RD001.D14.C2,0.321468,0.106061,0.182913,0.164841,0.065170,0.012778,0.093830,0.002008,0.008945,0.027747,0.014239
LAMI.RD001.D28.C1,0.399528,0.117766,0.178336,0.108318,0.113717,0.015860,0.025645,0.001687,0.016872,0.014172,0.008099
LAMI.RD001.D28.C2,0.359699,0.165254,0.197897,0.070308,0.143283,0.015223,0.019617,0.002197,0.006434,0.013026,0.007062
LAMI.RD007.D14.C1,0.534908,0.307128,0.043561,0.026841,0.053828,0.005867,0.011440,0.000000,0.001027,0.010854,0.004547
LAMI.RD007.D14.C2,0.554562,0.241523,0.058416,0.025894,0.078144,0.009710,0.021732,0.000000,0.000000,0.010018,0.000000
LAMI.RD004.D0.C1,0.475093,0.124978,0.235597,0.102819,0.012409,0.018436,0.003014,0.000886,0.005673,0.010636,0.010459
LAMI.RD006.D14.C2,0.564878,0.128593,0.152351,0.021764,0.069114,0.025918,0.002492,0.005483,0.021931,0.006978,0.000498


In [1124]:
merged_tbl_relative.columns

Index([' g__Cutibacterium', ' g__Staphylococcus', ' g__Corynebacterium',
       ' g__Streptococcus', ' g__Lawsonella', ' g__Micrococcus',
       ' g__Haemophilus', ' g__Kocuria', ' g__Lactobacillus',
       ' g__Veillonella', ' g__Rothia'],
      dtype='object')

In [1125]:
# Save as biom for mmvec
merged_tbl_relative_transposed = merged_tbl_relative.T
output_path = f'../Data/multi-omics/16S_MERGED-Avg-tbl_metaB-matched_relative_{skin_group}.biom'
save_as_biom(merged_tbl_relative_transposed, output_path)

In [1126]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_MERGED_tbl = metaB_tbl[metaB_tbl.index.isin(merged_tbl_relative.index)]

# Remove the 'group' column (only for VIP table)
# metaB_MERGED_tbl = metaB_MERGED_tbl.drop(columns=['group'])
metaB_MERGED_tbl.index.name = None

In [1127]:
# Sort columns by total sum (descending order)
metaB_MERGED_tbl = metaB_MERGED_tbl[metaB_MERGED_tbl.sum().sort_values(ascending=False).index]
metaB_MERGED_tbl

,"DL-Panthenol (mz: 206.1384472, RT: 1.1194918)","Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","Cyclo(his-pro) (mz: 235.1186308, RT: 0.6402779)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)","(Iso)leucine (mz: 276.143683, RT: 0.6789317)",...,"Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","Phenylbenzimidazole sulfonic acid (mz: 275.0481837, RT: 2.761023)","xylometazoline (mz: 245.2008949, RT: 4.3418465)","Nobiletin (mz: 403.138254, RT: 5.258404)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)"
LAMI.RD001.D0.C1,1473068.10,2987383.80,1942604.50,2025607.40,970487.50,567510.06,611709.700,302670.840,125692.3300,62007.1100,...,29574.6480,39859.7580,12406.82600,0.000,0.0000,0.000,0.0000,0.0,0.0,0.0
LAMI.RD001.D14.C1,504255.75,3269865.80,2762642.00,3296738.00,1496841.80,706262.44,1140515.800,651436.250,0.0000,87564.6100,...,48005.4060,34747.9200,43799.37000,0.000,17267.4320,0.000,0.0000,0.0,0.0,0.0
LAMI.RD001.D14.C2,3137497.20,2047720.40,2397076.20,3165493.20,1258087.60,717058.00,263219.340,218272.830,31278.8980,200700.7500,...,11624.8200,0.0000,9307.76500,0.000,0.0000,0.000,0.0000,0.0,0.0,0.0
LAMI.RD001.D0.C2,4475920.50,4255321.50,3177169.00,3035873.00,1358777.60,723508.56,944232.100,568705.940,76490.3800,107037.7500,...,24327.7580,23879.5450,22472.37500,0.000,0.0000,0.000,0.0000,0.0,0.0,0.0
LAMI.RD001.D28.C1,374392.06,5109647.00,3612457.50,3403914.80,1802260.60,1300446.40,269075.660,835043.000,42245.0160,127635.4450,...,38693.2970,51941.2540,88350.94000,36139.850,2983.9258,0.000,0.0000,0.0,0.0,0.0
LAMI.RD001.D28.C2,242698.45,3283120.20,2124453.20,2050733.50,927238.75,508464.70,424329.250,431731.720,18491.5880,69422.5550,...,62305.8670,39545.8050,31103.10500,0.000,0.0000,0.000,0.0000,0.0,0.0,0.0
LAMI.RD006.D14.C1,17990890.00,2010665.90,1216507.20,873745.75,335306.12,195284.60,7236.388,81698.730,12493.9800,36073.1170,...,0.0000,7931.4785,0.00000,0.000,36492.6170,0.000,0.0000,0.0,0.0,0.0
LAMI.RD006.D14.C2,18368216.00,1987542.20,1573511.80,1288035.90,547077.70,332766.40,15614.556,123767.820,6616.6740,56982.7580,...,12228.7720,25884.8440,4269.93550,11288.382,59558.3240,0.000,0.0000,0.0,0.0,0.0
LAMI.RD004.D28.C2,192005.80,481245.75,550417.06,214140.56,97621.54,66863.65,88451.336,130940.270,0.0000,3837.1455,...,0.0000,0.0000,0.00000,20156.625,0.0000,0.000,0.0000,0.0,0.0,0.0
LAMI.RD007.D14.C1,592174.94,1282058.00,1872823.50,1059865.60,472200.03,172225.81,339311.470,74992.780,107611.1200,50051.3500,...,8401.3790,0.0000,6149.22800,0.000,0.0000,0.000,0.0000,0.0,0.0,0.0


In [1128]:
# Convert to relative abundance (row-wise normalization)
metaB_MERGED_tbl_relative = metaB_MERGED_tbl.div(metaB_MERGED_tbl.sum(axis=1), axis=0)
metaB_MERGED_tbl_relative

,"DL-Panthenol (mz: 206.1384472, RT: 1.1194918)","Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)","Tyrosine (mz: 182.0809391, RT: 0.6720225)","(Iso)leucine (mz: 132.1017009, RT: 0.71511286)","Phenylalanine (mz: 166.086109, RT: 0.9777797)","Tryptophan (mz: 205.0969182, RT: 2.0760007)","C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)","Cyclo(his-pro) (mz: 235.1186308, RT: 0.6402779)","NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)","(Iso)leucine (mz: 276.143683, RT: 0.6789317)",...,"Urocanic acid (mz: 139.0410336, RT: 0.6406374)","Urocanic acid (mz: 139.0587028, RT: 0.6315576)","Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)","Sorbitane Monooleate (mz: 428.3728208, RT: 6.1563983)","Phenylbenzimidazole sulfonic acid (mz: 275.0481837, RT: 2.761023)","xylometazoline (mz: 245.2008949, RT: 4.3418465)","Nobiletin (mz: 403.138254, RT: 5.258404)","Gln-C14:0 (mz: 357.2754847, RT: 5.845114)","Sinensetin (mz: 373.127569, RT: 5.0247345)","3-O-methylated flavonoids (mz: 433.1489336, RT: 5.410203)"
LAMI.RD001.D0.C1,0.130990,0.265648,0.172743,0.180123,0.086299,0.050465,0.054395,0.026914,0.011177,0.005514,...,0.002630,0.003544,0.001103,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.0
LAMI.RD001.D14.C1,0.035618,0.230963,0.195136,0.232861,0.105728,0.049886,0.080559,0.046013,0.000000,0.006185,...,0.003391,0.002454,0.003094,0.000000,0.001220,0.00000,0.000000,0.0,0.0,0.0
LAMI.RD001.D14.C2,0.231220,0.150908,0.176654,0.233283,0.092716,0.052844,0.019398,0.016086,0.002305,0.014791,...,0.000857,0.000000,0.000686,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.0
LAMI.RD001.D0.C2,0.236712,0.225045,0.168027,0.160554,0.071860,0.038263,0.049936,0.030076,0.004045,0.005661,...,0.001287,0.001263,0.001188,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.0
LAMI.RD001.D28.C1,0.021648,0.295443,0.208874,0.196816,0.104208,0.075193,0.015558,0.048283,0.002443,0.007380,...,0.002237,0.003003,0.005108,0.002090,0.000173,0.00000,0.000000,0.0,0.0,0.0
LAMI.RD001.D28.C2,0.023498,0.317870,0.205688,0.198551,0.089775,0.049229,0.041083,0.041800,0.001790,0.006721,...,0.006032,0.003829,0.003011,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.0
LAMI.RD006.D14.C1,0.779160,0.087079,0.052685,0.037841,0.014522,0.008457,0.000313,0.003538,0.000541,0.001562,...,0.000000,0.000344,0.000000,0.000000,0.001580,0.00000,0.000000,0.0,0.0,0.0
LAMI.RD006.D14.C2,0.745155,0.080630,0.063834,0.052253,0.022194,0.013500,0.000633,0.005021,0.000268,0.002312,...,0.000496,0.001050,0.000173,0.000458,0.002416,0.00000,0.000000,0.0,0.0,0.0
LAMI.RD004.D28.C2,0.098307,0.246397,0.281812,0.109640,0.049982,0.034234,0.045287,0.067041,0.000000,0.001965,...,0.000000,0.000000,0.000000,0.010320,0.000000,0.00000,0.000000,0.0,0.0,0.0
LAMI.RD007.D14.C1,0.095579,0.206929,0.302280,0.171066,0.076215,0.027798,0.054766,0.012104,0.017369,0.008078,...,0.001356,0.000000,0.000993,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.0


In [1129]:
# Save as biom for mmvec
metaB_MERGED_tbl_relative_transposed = metaB_MERGED_tbl_relative.T
output_path = f'../Data/multi-omics/metaB_MERGED-Avg-tbl_matched_relative_{skin_group}.biom'
save_as_biom(metaB_MERGED_tbl_relative_transposed, output_path)

In [1130]:
metaB_MERGED_tbl_relative.columns

Index(['DL-Panthenol (mz: 206.1384472, RT: 1.1194918)',
       'Pyroglutamic acid (mz: 130.0497016, RT: 0.64796996)',
       'Tyrosine (mz: 182.0809391, RT: 0.6720225)',
       '(Iso)leucine (mz: 132.1017009, RT: 0.71511286)',
       'Phenylalanine (mz: 166.086109, RT: 0.9777797)',
       'Tryptophan (mz: 205.0969182, RT: 2.0760007)',
       'C19H36O4 Fatty alcohol (mz: 311.2573967, RT: 7.4214296)',
       'Cyclo(his-pro) (mz: 235.1186308, RT: 0.6402779)',
       'NCGC00380271-01 (mz: 505.2636006, RT: 3.5891602)',
       '(Iso)leucine (mz: 276.143683, RT: 0.6789317)',
       'Nicotine (mz: 163.1224157, RT: 0.634317)',
       'Uridine (mz: 245.0765944, RT: 0.65536463)',
       'Paracetamol (mz: 152.0704373, RT: 1.6221956)',
       'N-Oleoylethanolamine (mz: 326.3050565, RT: 7.1323557)',
       'Urocanic acid (mz: 139.0410336, RT: 0.6406374)',
       'Urocanic acid (mz: 139.0587028, RT: 0.6315576)',
       'Glutamyltyrosine (mz: 311.1238575, RT: 1.2036344)',
       'Sorbitane Monooleate 